# 01 - Connectivity PCA

**Purpose**: reduce the eLife-grouped connectivity matrix (subjects x regions)
to a few principal components and inspect which regions drive each.

**Input**: ``data.FCDGdf_wide`` -- connectomics filtered to matched subjects,
eLife-grouped, wide format.

**Caveat**: N=few matched subjects. PCA results are shown to
validate the pipeline end-to-end; interpretation at small N is
suggestive, not statistical. Re-run as N grows.

Figures written to ``example_output/``:
  - ``01_scree.png``
  - ``01_loadings_all.png``
  - ``01_loadings_important.png``
  - ``01_loadings_heatmap.png``
  - ``01_connectivity_heatmap.png``


In [ ]:
# parameters
# Tweak these values to re-render with different settings. No other edits needed.
N_COMPONENTS = 3       # how many principal components to extract
TOP_N_REGIONS = 10     # how many regions to mark as "important" per component
FIGSIZE_SCREE = (8, 6)
FIGSIZE_LOADINGS = (30, 30)
FIGSIZE_LOADINGS_FILTERED = (20, 10)
FIGSIZE_HEATMAP = (6, 10)
FIGSIZE_SUBJECT_REGION = (24, 4)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import endpoint_ck_analysis  # Soft-imports mousedb.region_priors with a frozen fallback
from endpoint_ck_analysis import SKILLED_REACHING, ordered_hemisphere_columns
from endpoint_ck_analysis.config import EXAMPLE_OUTPUT_DIR
from endpoint_ck_analysis.data_loader import load_all
from endpoint_ck_analysis.helpers.plotting import stamp_version

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

data = load_all()  # Reads from cache produced by 00_setup


## 1. Prepare the matrix and run PCA

Apply the canonical region ordering from ``mousedb.region_priors`` so
every subsequent plot shares the same row/column order -- predicted-important
regions sit together at the top.


In [ ]:
X = data.FCDGdf_wide.fillna(0)  # PCA cannot handle NaNs; fill with 0

# Canonical column order: predicted importance for skilled reaching, both/left/right per region.
canonical_cols = ordered_hemisphere_columns(
    SKILLED_REACHING, available=X.columns.tolist()
)
X = X[canonical_cols]  # Reorder so every downstream plot inherits this order

scaler = StandardScaler()            # Z-score each region's counts across subjects
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=N_COMPONENTS)
scores = pca.fit_transform(X_scaled)

eigen_summary = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(N_COMPONENTS)],
    'Eigenvalue': pca.explained_variance_,
    'Variance': pca.explained_variance_ratio_,
    'Cumulative': np.cumsum(pca.explained_variance_ratio_),
})
print(eigen_summary)


## 2. Scree plot

Variance explained by each principal component. If PC1 and PC2 together
explain most of the variance, a 2D plot captures the dominant structure.


In [ ]:
fig = plt.figure(figsize=FIGSIZE_SCREE)
plt.bar(range(1, N_COMPONENTS + 1), pca.explained_variance_ratio_)
plt.xticks(range(1, N_COMPONENTS + 1), eigen_summary['Component'])
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained')
plt.title('Connectivity PCA: Variance per PC')
stamp_version(fig, label='01 scree')
plt.savefig(EXAMPLE_OUTPUT_DIR / '01_scree.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Loadings: all regions, in canonical order

Horizontal bars per PC. A region with a long bar (positive or negative)
contributes strongly to that component's differentiation between subjects.


In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    columns=X.columns,
    index=[f'PC{i+1}' for i in range(N_COMPONENTS)],
)

fig, axes = plt.subplots(1, N_COMPONENTS, figsize=FIGSIZE_LOADINGS, sharey=True)
for i, pc in enumerate(loadings.index):
    ordered = loadings.loc[pc, canonical_cols]  # In canonical order, not sorted by value
    axes[i].barh(ordered.index, ordered.values)
    axes[i].set_xlabel('Loading')
    axes[i].set_title(f'{pc} ({pca.explained_variance_ratio_[i]:.1%} var)')
    axes[i].axvline(0, color='black', linewidth=0.5)
    axes[i].invert_yaxis()  # Highest-priority regions on top
stamp_version(fig, label='01 loadings (all)')
plt.savefig(EXAMPLE_OUTPUT_DIR / '01_loadings_all.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Identify the top-loading regions per PC

For each component, pick the ``TOP_N_REGIONS`` regions by absolute
loading magnitude. The union across components defines the
"important regions" that carry forward into PLS as X-block features.


In [ ]:
important_regions = {}
for pc in loadings.index:
    region_importance = loadings.loc[pc].abs()
    top_regions = region_importance.nlargest(TOP_N_REGIONS)
    important_regions[pc] = top_regions
    print(f"\nTop {TOP_N_REGIONS} regions on {pc} by |loading|:")
    print(top_regions)

all_important_regions = set()
for pc, series in important_regions.items():
    all_important_regions.update(series.index)
print(f"\n{len(all_important_regions)} unique important regions:")
print(sorted(all_important_regions))


## 5. Loadings: important regions only (canonical order)


In [ ]:
important_cols_ordered = ordered_hemisphere_columns(
    SKILLED_REACHING, available=list(all_important_regions)
)
fig, axes = plt.subplots(1, N_COMPONENTS, figsize=FIGSIZE_LOADINGS_FILTERED)
for i, pc in enumerate(loadings.index):
    pc_top = loadings.loc[pc, important_cols_ordered]
    axes[i].barh(pc_top.index, pc_top.values)
    axes[i].axvline(0, color='black', linewidth=0.5)
    axes[i].set_xlabel('Loading')
    axes[i].set_title(f'{pc} (important regions only)')
    axes[i].invert_yaxis()
stamp_version(fig, label='01 loadings (important)')
plt.savefig(EXAMPLE_OUTPUT_DIR / '01_loadings_important.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Loadings heatmap


In [ ]:
important_loadings = loadings[important_cols_ordered].T  # regions as rows, PCs as columns
fig = plt.figure(figsize=FIGSIZE_HEATMAP)
ax = sns.heatmap(important_loadings, cmap='RdBu_r', center=0,
                 cbar_kws={'label': 'Loading'})
ax.invert_yaxis()
plt.title('Connectivity loadings (important regions only)')
stamp_version(fig, label='01 loadings heatmap')
plt.savefig(EXAMPLE_OUTPUT_DIR / '01_loadings_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Subject x region cell-count heatmap

Each row is a subject, each column is a region (canonical order). The
red dashed line marks the boundary between predicted-important regions
(above) and kept-for-contrast regions (below).


In [ ]:
fig = plt.figure(figsize=FIGSIZE_SUBJECT_REGION)
ax = sns.heatmap(X, cmap='viridis', cbar_kws={'label': 'Cell count'})

high_priority_region_names = set(
    SKILLED_REACHING.ordered_regions[:SKILLED_REACHING.high_priority_cutoff]
)
cutoff_cols = sum(
    1 for c in canonical_cols if c.rsplit('_', 1)[0] in high_priority_region_names
)
ax.axvline(cutoff_cols, color='red', linestyle='--', linewidth=1.5)
plt.title('Subject x region cell counts (canonical order; red = priority cutoff)')
stamp_version(fig, label='01 cell counts')
plt.savefig(EXAMPLE_OUTPUT_DIR / '01_connectivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Export the important-regions list for the PLS notebook

The set is written to the cache as ``important_regions.parquet`` so
``04_pls_variants`` can read it without recomputing.


In [ ]:
from endpoint_ck_analysis.config import CACHE_DIR
pd.Series(sorted(all_important_regions), name='region_hemi').to_frame().to_parquet(
    CACHE_DIR / 'important_regions.parquet', index=False
)
print(f'Wrote {len(all_important_regions)} important regions to cache.')


<!-- INTERPRETATION_BLOCK -->
## How to read this notebook's output

<details>
<summary>What the connectivity PCA outputs tell you (click to expand)</summary>

**Scree plot**: how variance partitions across PCs.

- PC1 alone capturing >50% of variance = one dominant axis of injury
  variation; mice differ mostly in one coherent connectivity pattern.
- PC1 + PC2 together capturing >80% = most structure is 2D; the
  3D / higher PCs are noise.
- Variance spread evenly across PCs = no dominant pattern; subjects
  differ in many independent ways.

**Loading bars (per-PC)**: which connectivity regions drive each component.

- Top regions on PC1 are where injuries vary MOST across the matched
  cohort. They are NOT necessarily the regions that matter for reaching --
  they are just the ones where sparing differs most between mice. PLS
  (notebook 04) is what tells you which of those couple to kinematics.
- A region with a long bar in one direction (positive or negative)
  contributes strongly to that PC. The sign is arbitrary on its own;
  what matters is the magnitude and the relative ordering.

**Subject x region heatmap with red dashed line**: visual sanity check.
The red line is the predicted-importance cutoff from
`SKILLED_REACHING.high_priority_cutoff`. If subjects' bright cells cluster
to the LEFT of the line, the prior ordering aligns with what's varying
most in the data; if they're spread evenly or favor the RIGHT, the
literature-derived prior may be missing something the data sees.

At small N every PC loading is noisy. Treat the top-10 region list as
suggestive, not definitive. Re-run as N grows.

</details>
